# YOLOv8 Training on Local Machine (Save in Workspace)

In [ ]:
import sys
import subprocess

print("Python executable:", sys.executable)
print("Installing PyTorch with CUDA 12.1...\n")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--progress-bar",
        "on",
        "-v",
        "--index-url",
        "https://download.pytorch.org/whl/cu121",
        "torch",
        "torchvision",
        "torchaudio"
    ],
    check=True
)

print("\nInstalling Ultralytics and PyYAML...\n")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--progress-bar",
        "on",
        "-v",
        "ultralytics==8.4.22",
        "PyYAML"
    ],
    check=True
)

print("\nInstallation complete.")

In [1]:
import sys
import subprocess

packages = [
"torch",
"torchvision",
"torchaudio",
"ultralytics",
"PyYAML",
]

print("Python executable:", sys.executable)
print("Uninstalling packages...")

subprocess.run(
[sys.executable, "-m", "pip", "uninstall", "-y", *packages],
check=True
)

print("Optional: clearing pip cache...")
subprocess.run([sys.executable, "-m", "pip", "cache", "purge"], check=False)

print("Uninstall complete. Restart kernel.")

Python executable: e:\capstone\cleanopsai-ai\.venv\Scripts\python.exe
Uninstalling packages...
Optional: clearing pip cache...
Uninstall complete. Restart kernel.


# Check Local Hardware (GPU/CPU)

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

import ultralytics
import torch

print("Workspace:", Path.cwd())
print("Ultralytics version:", ultralytics.__version__)
print("PyTorch version:", torch.__version__)
print("Torch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

nvidia_smi = shutil.which("nvidia-smi")
print("nvidia-smi path:", nvidia_smi)
if nvidia_smi:
    subprocess.run([nvidia_smi, "-L"], check=False)
else:
    print("nvidia-smi not found (this is normal on CPU-only machines).")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
else:
    print("No GPU detected. Notebook will run on CPU.")

In [ ]:
from pathlib import Path
import subprocess

dataset_dir = Path.cwd() / "ppe-detection-dataset"
if dataset_dir.exists():
    print(f"Dataset already exists at: {dataset_dir}")
else:
    subprocess.run(
        [
            "git",
            "clone",
            "https://huggingface.co/datasets/jhboyo/ppe-dataset",
            str(dataset_dir),
        ],
        check=True,
    )
    print(f"Dataset cloned to: {dataset_dir}")

In [ ]:
import yaml
from pathlib import Path

candidate_roots = [
    Path.cwd() / "ppe-detection-dataset",
    Path.cwd() / "ppe-dataset",
]

dataset_root = next((p for p in candidate_roots if p.exists()), None)
if dataset_root is None:
    raise FileNotFoundError(
        "Dataset folder not found in workspace. Run the clone cell first, then rerun this cell."
    )

train_dir = dataset_root / "train" / "images"
val_dir_valid = dataset_root / "valid" / "images"
val_dir_val = dataset_root / "val" / "images"

if val_dir_valid.exists():
    val_dir = val_dir_valid
    val_key = "valid/images"
elif val_dir_val.exists():
    val_dir = val_dir_val
    val_key = "val/images"
else:
    raise FileNotFoundError(
        f"Validation images folder not found. Checked: {val_dir_valid} and {val_dir_val}"
    )

if not train_dir.exists():
    raise FileNotFoundError(f"Training images folder not found: {train_dir}")

data = {
    "path": str(dataset_root),
    "train": "train/images",
    "val": val_key,
    "names": {
        0: "helmet",
        1: "vest",
        2: "person",
    },
}

data_yaml = Path.cwd() / "dataset.yaml"
with open(data_yaml, "w", encoding="utf-8") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print(f"dataset.yaml created at: {data_yaml}")
print(f"Using dataset root: {dataset_root}")
print(f"Using train path: {train_dir}")
print(f"Using val path: {val_dir}")

In [ ]:
!yolo settings

In [ ]:
import os
import shutil
from pathlib import Path

import torch
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not available in PyTorch. Run Cell 2 to install CUDA PyTorch, restart kernel, then rerun checks."
    )

torch.backends.cudnn.benchmark = True
device = 0
imgsz = 640
batch = -1
workers = min(4, os.cpu_count() or 2)
amp = True
train_name = "train_local_gpu"

project_dir = Path.cwd() / "runs"
project_dir.mkdir(parents=True, exist_ok=True)

data_yaml = Path.cwd() / "dataset.yaml"
if not data_yaml.exists():
    raise FileNotFoundError(f"{data_yaml} not found. Run the dataset.yaml cell first.")

print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"Training outputs will be saved to: {project_dir}")

model = YOLO("yolov8n.pt")
results = model.train(
    data=str(data_yaml),
    epochs=30,
    imgsz=imgsz,
    device=device,
    batch=batch,
    workers=workers,
    project=str(project_dir),
    name=train_name,
    exist_ok=True,
    cache=True,
    amp=amp,
    save=True,
    plots=True,
    verbose=True,
)

best_src = Path(results.save_dir) / "weights" / "best.pt"
best_dst = Path.cwd() / "best_local_gpu.pt"
if best_src.exists():
    shutil.copy2(best_src, best_dst)
    print(f"Training finished. Save dir: {results.save_dir}")
    print(f"Best model copied to: {best_dst}")
else:
    print(f"Training finished but best.pt not found at: {best_src}")

In [ ]:
import glob
import os
from pathlib import Path

base_runs = Path.cwd() / "runs"
models = glob.glob(str(base_runs / "train*" / "weights" / "*.pt"))
models = sorted(models, key=os.path.getmtime, reverse=True)

if not models:
    print(f"No model files found under: {base_runs}")
else:
    print("Latest model files:")
    for m in models[:20]:
        print(m)

best_local = Path.cwd() / "best_local_gpu.pt"
if best_local.exists():
    print(f"Workspace best model: {best_local}")